# 04 — 权限（Permissions）

**来源:** [LangChain Docs — Permissions](https://docs.langchain.com/oss/python/deepagents/permissions)

使用声明式权限规则控制 Agent 对文件系统的读写访问。通过 `permissions=` 参数传入规则列表，Agent 的内置文件系统工具会遵循这些规则。

> 注意：权限需要 `deepagents>=0.5.2`。仅适用于内置文件系统工具（`ls`、`read_file`、`glob`、`grep`、`write_file`、`edit_file`），不覆盖自定义工具、MCP 工具或沙箱 Backend 的 `execute` 工具。

## 1. 基本用法

传入 `FilesystemPermission` 规则列表。规则按声明顺序评估，**首个匹配**决定结果。没有规则匹配时默认**允许**。

In [ ]:
# 基本用法：只读 Agent，拒绝所有写入

from deepagents import FilesystemPermission, create_deep_agent
from dotenv import load_dotenv
load_dotenv()

agent = create_deep_agent(
    model="deepseek-v4-flash",
    permissions=[
        FilesystemPermission(
            operations=["write"],   # 禁止写入操作
            paths=["/**"],          # 所有路径
            mode="deny",
        ),
    ],
)


## 2. 规则结构

每个 `FilesystemPermission` 包含三个字段：

| 字段 | 类型 | 说明 |
|------|------|------|
| `operations` | `list["read" \| "write"]` | 操作类型。`"read"` 覆盖 `ls`、`read_file`、`glob`、`grep`；`"write"` 覆盖 `write_file`、`edit_file` |
| `paths` | `list[str]` | 文件路径的 glob 模式（如 `["/workspace/**"]`）。支持 `**` 递归匹配和 `{a,b}` 选择 |
| `mode` | `"allow" \| "deny"` | 允许或拒绝。默认 `"allow"` |

> 规则使用**首个匹配胜出**评估机制。第一个匹配的规则决定结果。无匹配时默认允许。

## 3. 常见场景

### 3.1 隔离到工作区目录

只允许在 `/workspace/` 下读写，拒绝其他所有路径。

In [ ]:
# 隔离到工作区：仅允许 /workspace/ 下的读写

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    permissions=[
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/workspace/**"],  # 仅工作区路径允许
            mode="allow",
        ),
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/**"],             # 其他所有路径拒绝
            mode="deny",
        ),
    ],
)

### 3.2 保护特定文件

拒绝访问 `.env` 和 `examples/` 目录，工作区其他路径允许。

In [ ]:
# 保护特定文件：拒绝 .env 和 examples/，其余 workspace 允许

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    permissions=[
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/workspace/.env", "/workspace/examples/**"],  # 先拒绝敏感路径
            mode="deny",
        ),
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/workspace/**"],  # 允许工作区
            mode="allow",
        ),
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/**"],             # 拒绝其余所有
            mode="deny",
        ),
    ],
)

### 3.3 只读记忆

Agent 可以读取记忆文件但无法修改。适用于组织政策或共享知识库。

In [ ]:
# 只读记忆：允许读取 /memories/ 和 /policies/，拒绝写入

from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,),
            ),
            "/policies/": StoreBackend(
                namespace=lambda rt: (rt.context.org_id,),
            ),
        },
    ),
    permissions=[
        FilesystemPermission(
            operations=["write"],
            paths=["/memories/**", "/policies/**"],  # 拒绝写入记忆和政策文件
            mode="deny",
        ),
    ],
)

### 3.4 拒绝所有访问

作为严格基线，在此之上叠加更具体的允许规则。

In [ ]:
# 拒绝所有文件系统访问

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    permissions=[
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/**"],
            mode="deny",
        ),
    ],
)

## 4. 规则排序陷阱

由于**首个匹配胜出**，规则顺序非常重要。具体规则必须放在更宽泛规则之前。

In [ ]:
# 规则排序：正确与错误示例

# 正确：先拒绝 .env，再允许 workspace，最后拒绝其余
correct_permissions = [
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/.env"],  # 具体规则先出现
        mode="deny",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/**"],
        mode="allow",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/**"],
        mode="deny",
    ),
]

# 错误：/workspace/** 匹配 /workspace/.env，拒绝规则永远不会触发
incorrect_permissions = [
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/**"],  # 这个先匹配了 .env 并允许
        mode="allow",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/.env"],
        mode="deny",  # 永远不会到达这里
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/**"],
        mode="deny",
    ),
]

## 5. Subagent 权限

Subagent 默认**继承**父 Agent 的权限规则。要赋予 Subagent 不同权限，在其 spec 中设置 `permissions` 字段——这会**完全替换**父 Agent 的规则。

In [ ]:
# Subagent 使用自己独立的权限规则

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
    permissions=[
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/workspace/**"],
            mode="allow",
        ),
        FilesystemPermission(
            operations=["read", "write"],
            paths=["/**"],
            mode="deny",
        ),
    ],
    subagents=[
        {
            "name": "auditor",
            "description": "Read-only code reviewer",
            "system_prompt": "Review the code for issues.",
            "permissions": [   # 独立权限，不继承父 Agent
                FilesystemPermission(
                    operations=["write"],
                    paths=["/**"],
                    mode="deny",
                ),
                FilesystemPermission(
                    operations=["read"],
                    paths=["/workspace/**"],
                    mode="allow",
                ),
                FilesystemPermission(
                    operations=["read"],
                    paths=["/**"],
                    mode="deny",
                ),
            ],
        }
    ],
)

## 6. CompositeBackend 与权限

使用 `CompositeBackend` 且默认 backend 为沙箱时，权限路径**必须限定**在已知的路由前缀下。沙箱支持任意命令执行，仅靠路径限制无法阻止通过 shell 命令的文件系统访问。

In [ ]:
# CompositeBackend 中权限路径必须限定在路由前缀下

from deepagents.backends import CompositeBackend

composite = CompositeBackend(
    default=sandbox,
    routes={"/memories/": memories_backend},
)

# 正确：权限限定在 /memories/ 路由下
agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=composite,
    permissions=[
        FilesystemPermission(
            operations=["write"],
            paths=["/memories/**"],  # 限定在路由前缀下
            mode="deny",
        ),
    ],
)

In [ ]:
# 错误：包含路由外的路径会引发 NotImplementedError

try:
    create_deep_agent(
        model="deepseek-v4-flash",
        backend=composite,
        permissions=[
            FilesystemPermission(
                operations=["write"],
                paths=["/workspace/**"],  # /workspace/ 不在任何路由中，触及沙箱默认
                mode="deny",
            ),
        ],
    )
except NotImplementedError:
    print("NotImplementedError: 沙箱默认不支持路径权限")

# 同样错误：/** 同时覆盖路由和默认
try:
    create_deep_agent(
        model="deepseek-v4-flash",
        backend=composite,
        permissions=[
            FilesystemPermission(
                operations=["read"],
                paths=["/**"],  # 触及沙箱默认
                mode="deny",
            ),
        ],
    )
except NotImplementedError:
    print("NotImplementedError: /** 触及沙箱默认，不受支持")

# 结论：沙箱默认下，权限路径必须限定在已知路由前缀内

## 最佳实践总结

| 场景 | 推荐方式 |
|------|----------|
| 只读 Agent | `operations=["write"], paths=["/**"], mode="deny"` |
| 隔离工作区 | 先 allow workspace，再 deny 其余 |
| 保护敏感文件 | 具体 deny 规则放在 allow 规则**之前** |
| 组织级策略只读 | `operations=["write"], paths=["/policies/**"], mode="deny"` |
| Subagent 特殊权限 | 在 subagent spec 中设置独立 `permissions` |
| 沙箱默认 + CompositeBackend | 权限路径必须限定在路由前缀下 |

---
**参考链接**

- [Permissions — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/permissions)
- [Documentation Index](https://docs.langchain.com/llms.txt)
- [Deep Agents Customization](https://docs.langchain.com/oss/python/deepagents/customization)